In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [2]:
df = pd.read_csv("Fraud Detection.csv")

In [3]:
df.head()

,step,type,branch,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,unusuallogin,isFlaggedFraud,Acct type,Date of transaction,Time of day,isFraud,Column1,isFraud - Copy,DayOfWeek,DayOfWeek(new)
0,7,PAYMENT,Espana,1526.50,C1908839976,0.0,0.0,M1304522697,0.0,0.0,7,0,Savings,03-Jan-18,Afternoon,Safe,7168,0.0,3.0,Wednesday
1,7,PAYMENT,Honduras,1620.17,C70432899,0.0,0.0,M252297858,0.0,0.0,2,0,Savings,05-Jan-18,Night,Safe,7211,0.0,5.0,Friday
2,7,PAYMENT,Indonesia,5689.82,C332151172,0.0,0.0,M1430674428,0.0,0.0,3,0,Savings,07-Jan-18,Morning,Safe,7268,0.0,7.0,Sunday
3,7,PAYMENT,Estados Unidos,2211.07,C1148172078,0.0,0.0,M650835126,0.0,0.0,13,0,Savings,06-Jan-18,Afternoon,Safe,7325,0.0,6.0,Saturday
4,7,PAYMENT,Estados Unidos,824.63,C1142006014,0.0,0.0,M745868137,0.0,0.0,7,0,Savings,06-Jan-18,Morning,Safe,7326,0.0,6.0,Saturday


In [4]:
df.describe

<bound method NDFrame.describe of        step      type          branch     amount     nameOrig  oldbalanceOrg  \
0         7   PAYMENT          Espana    1526.50  C1908839976            0.0   
1         7   PAYMENT        Honduras    1620.17    C70432899            0.0   
2         7   PAYMENT       Indonesia    5689.82   C332151172            0.0   
3         7   PAYMENT  Estados Unidos    2211.07  C1148172078            0.0   
4         7   PAYMENT  Estados Unidos     824.63  C1142006014            0.0   
...     ...       ...             ...        ...          ...            ...   
10122     7  CASH_OUT  Estados Unidos  488632.10   C984363447            0.0   
10123     7  CASH_OUT  Estados Unidos  232784.28   C144560386            0.0   
10124     7  CASH_OUT  Estados Unidos  223310.57  C1677203878            0.0   
10125     7  CASH_OUT  Estados Unidos  302788.52  C2082418981            0.0   
10126     7  CASH_OUT  Estados Unidos   36600.63  C1157016655            0.0   

     

In [5]:
df.isnull().sum()

step                    0
type                    4
branch                  0
amount                  2
nameOrig                6
oldbalanceOrg           2
newbalanceOrig          0
nameDest                6
oldbalanceDest          1
newbalanceDest          2
unusuallogin            0
isFlaggedFraud          0
Acct type              10
Date of transaction     7
Time of day             2
isFraud                 0
Column1                 0
isFraud - Copy          2
DayOfWeek               7
DayOfWeek(new)          7
dtype: int64

------ Dropping the duplicates , not useful columns , and filling the columns with median and mode for numbers and text 

In [6]:
df = df.drop(columns=['nameOrig', 'nameDest', 'isFraud - Copy', 
                       'DayOfWeek', 'Column1', 'isFlaggedFraud'])


In [7]:
df.isnull().sum()

step                    0
type                    4
branch                  0
amount                  2
oldbalanceOrg           2
newbalanceOrig          0
oldbalanceDest          1
newbalanceDest          2
unusuallogin            0
Acct type              10
Date of transaction     7
Time of day             2
isFraud                 0
DayOfWeek(new)          7
dtype: int64

In [8]:
df["type"] = df["type"].fillna(df["type"].mode()[0])

In [9]:
df["Acct type"]= df["Acct type"].fillna(df["Acct type"].mode()[0])

In [10]:
df['Time of day'] = df['Time of day'].fillna(df['Time of day'].mode()[0])
df['DayOfWeek(new)'] = df['DayOfWeek(new)'].fillna(df['DayOfWeek(new)'].mode()[0])

In [11]:
df['amount'] = df['amount'].fillna(df['amount'].median())
df['oldbalanceOrg'] = df['oldbalanceOrg'].fillna(df['oldbalanceOrg'].median())
df['oldbalanceDest'] = df['oldbalanceDest'].fillna(df['oldbalanceDest'].median())
df['newbalanceDest'] = df['newbalanceDest'].fillna(df['newbalanceDest'].median())


In [12]:
df.isnull().sum()

step                   0
type                   0
branch                 0
amount                 0
oldbalanceOrg          0
newbalanceOrig         0
oldbalanceDest         0
newbalanceDest         0
unusuallogin           0
Acct type              0
Date of transaction    7
Time of day            0
isFraud                0
DayOfWeek(new)         0
dtype: int64

In [13]:
df = df.dropna(subset=['Date of transaction'])

In [14]:
df.isnull().sum()

step                   0
type                   0
branch                 0
amount                 0
oldbalanceOrg          0
newbalanceOrig         0
oldbalanceDest         0
newbalanceDest         0
unusuallogin           0
Acct type              0
Date of transaction    0
Time of day            0
isFraud                0
DayOfWeek(new)         0
dtype: int64

In [15]:
df.dtypes

step                     int64
type                       str
branch                     str
amount                 float64
oldbalanceOrg          float64
newbalanceOrig         float64
oldbalanceDest         float64
newbalanceDest         float64
unusuallogin             int64
Acct type                  str
Date of transaction        str
Time of day                str
isFraud                    str
DayOfWeek(new)             str
dtype: object

--- Encodin the Strings into text ------

In [16]:
encoder = LabelEncoder()

df["type"] = encoder.fit_transform(df["type"])


In [17]:
df["branch"]= encoder.fit_transform(df["branch"])

In [18]:
df["Acct type"]= encoder.fit_transform(df["Acct type"])

In [19]:
df["Date of transaction"]= encoder.fit_transform(df["Date of transaction"])

In [20]:
df["Time of day"]= encoder.fit_transform(df["Time of day"])

In [21]:
df["isFraud"]= encoder.fit_transform(df["isFraud"])

In [22]:
df["DayOfWeek(new)"]= encoder.fit_transform(df["DayOfWeek(new)"])

---- Features selection ----------

In [23]:
x = df.drop("isFraud",axis=1)

In [24]:
scaler = StandardScaler()
x = scaler.fit_transform(x)

----- Using elbow method to find the best cluster

In [25]:
wcss = []
for i in range(2, 11):
    model = KMeans(n_clusters=i, random_state=42, n_init=10)
    model.fit(x)
    wcss.append(model.inertia_)

In [26]:
model = KMeans(n_clusters=2, random_state=42, n_init=10)

labels = model.fit_predict(x)

In [27]:
print(labels)

[0 0 0 ... 0 0 0]


--- using silhuouette score to unserstand the quality of cluster ---- 

In [28]:
score = silhouette_score(x, labels)

print("Silhouette Score:", score)

Silhouette Score: 0.32186289714548694


In [29]:
import pickle
pickle.dump(encoder,open(r"C:\Users\kokch\45_days_Training\Machine_Learning\Fraud Detection\model\encoder.pkl","wb")) 

In [30]:
pickle.dump(scaler,open(r"C:\Users\kokch\45_days_Training\Machine_Learning\Fraud Detection\model\scaler.pkl","wb"))

In [31]:
pickle.dump(model,open(r"C:\Users\kokch\45_days_Training\Machine_Learning\Fraud Detection\model\model.pkl","wb"))